# Stage 2 — activation, one dataset per cohort

Parcel timeseries: one row per TR, one float32 column per parcel, four cohorts
in one tree separated by the `cohort=` partition key.

This notebook does four things and stops:

1. **inventory** what is on disk — cohorts, atlases, subject counts — without
   opening a file, and check it against what the configs say should be there;
2. **load one activation dataset per cohort** on a coarse atlas, small enough
   to hold all four at once;
3. **sanity-check** them: TR, censoring, empty parcels;
4. **join** participants curation, pipeline QC, and phenotype (age / sex /
   clinical) where it exists.

Everything after that is yours to direct.

### Running it

Python lives only inside the container, Jupyter included, and nothing heavy
belongs on a login node:

```bash
module load apptainer/1.4.5
salloc --account=rpp-aevans-ab --cpus-per-task=4 --mem=32G --time=3:00:00

export FMRIDECOMP_SIF=/project/6008063/tamires/singularity/fmri_decomp.sif
cd /project/6008063/tamires/DecomposingfMRI
apptainer exec --bind /project,/scratch,/home --pwd $PWD $FMRIDECOMP_SIF \
    jupyter lab --no-browser --ip=0.0.0.0 --port=8888
```

then tunnel to the compute node from your laptop. `notebooks/README.md` has the
full recipe and the arithmetic behind `--mem=32G`.

## Setup

`nbtools` is the loader module next to this notebook. It wraps
`fmri_decomposition.io` — every path it builds comes from there — and adds the
partition-pruned, column-projected reads a notebook needs. Read it: it is
short, and each function's docstring says what it refuses to do.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))          # notebooks/ on the path
import nbtools as nb

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except ImportError:                          # the container may not ship it
    HAVE_MPL = False
    print("matplotlib absent -- plots are skipped, tables still print")

# Parameters. ATLAS_COARSE is what a whole cohort is loaded at; ATLAS_FINE is
# what a single subject is loaded at. 111 parcels x every subject x every TR is
# not a notebook read.
ATLAS_COARSE = "yeo7"
ATLAS_FINE = "harvardoxford"

ROOT = nb.output_root()                      # FMRIDECOMP_OUTPUTS overrides
print("output_root:", ROOT)

In [ ]:
# What each cohort was configured to be. TR and the window grid are here
# because they are what makes a cross-cohort comparison hard later: a window of
# fixed duration is a different number of samples in each of these.
nb.cohort_configs()

## 1. What is actually on disk

Not what the configs asked for — what got written. This is a glob over
directory names at fixed depth: no file is opened, nothing is stat-ed, and it
stays cheap at thousands of leaves. It also reports abandoned `.tmp.<pid>`
shards, which `pyarrow.dataset` cannot see at all (wrong extension) and which
would otherwise look exactly like a subject that never ran.

In [ ]:
inv_act = nb.inventory("activation")
print(f"{len(inv_act)} activation shard(s)")
nb.inventory_summary(inv_act, "activation")

In [ ]:
# Subjects per cohort per atlas, as a grid -- a hole here is a stage-2 gap.
inv_act.pivot_table(index="cohort", columns="atlas", values="sub",
                    aggfunc="nunique", fill_value=0)

In [ ]:
# On disk vs. the participants table: both directions matter. A subject in the
# table with no shard failed or was never run; a shard with no table row means
# nobody has said whether it belongs in the analysis.
rows = []
for cohort in sorted(inv_act["cohort"].unique()):
    on_disk = set(inv_act.loc[inv_act["cohort"] == cohort, "sub"])
    part = nb.load_participants(cohort)
    listed = set(part["sub"])
    analysable = set(part.loc[~part["excluded"], "sub"])
    rows.append({
        "cohort": cohort,
        "subs_on_disk": len(on_disk),
        "subs_in_table": len(listed),
        "excluded_in_table": int(part["excluded"].sum()),
        "in_table_not_on_disk": len(analysable - on_disk),
        "on_disk_not_in_table": len(on_disk - listed),
        "tasks_on_disk": inv_act.loc[inv_act["cohort"] == cohort, "task"].nunique(),
    })
pd.DataFrame(rows)

### The partition key that comes back null

A hive key is recovered from the path **relative to the dataset root**, so a key
at or above the root is not in the path any more and pyarrow fills the column
with nulls. One dataset per atlas means the root *is* the `atlas=` directory —
so `atlas` is always one of those columns, and a filter on it would match
nothing rather than raise.

`nbtools` opens at the atlas root (which is what keeps the schema homogeneous)
and backfills the keys the root swallowed. Worth seeing once:

In [ ]:
import pyarrow.dataset as pads
from fmri_decomposition.io import activation_root, open_dataset

raw = open_dataset(activation_root(ROOT, ATLAS_COARSE), stage="activation")
raw_head = raw.head(2, columns=["t", "atlas", "cohort", "sub"]).to_pandas()
print("opened by hand at the atlas root -- note `atlas`:")
print(raw_head)
print("\nrows matching a filter on that null column:",
      raw.to_table(filter=pads.field("atlas") == ATLAS_COARSE).num_rows)
print("\nthrough nbtools.load_activation:")
print(nb.load_activation(ATLAS_COARSE, cohort=inv_act['cohort'].iloc[0])
        .head(2)[["t", "atlas", "cohort", "sub"]])

## 2. One dataset per cohort

`yeo7` is 7 parcels, so a whole cohort — every subject, every TR — is tens of
MB and all four fit in memory together. `load_activation` prunes to one
`cohort=` directory and projects the columns; the parcel columns are opt-in
via `with_parcels=True`.

In [ ]:
COHORTS = sorted(inv_act["cohort"].unique())

act = {}
for cohort in COHORTS:
    act[cohort] = nb.load_activation(ATLAS_COARSE, cohort=cohort, with_parcels=True)

pd.DataFrame([{
    "cohort": c,
    "rows": len(df),
    "subs": df["sub"].nunique(),
    "tasks": df["task"].nunique(),
    "parcel_cols": len([x for x in df.columns
                        if x not in nb.ACTIVATION_META_COLUMNS
                        and x not in ("atlas", "cohort", "task", "sub")]),
    "mem_MB": round(nb.mem_mb(df), 1),
} for c, df in act.items()])

In [ ]:
# The shape of one of them. `ses`/`run`/`acq`/`run_key` are columns, not path
# keys -- since every leaf is `data.parquet`, they are the only record of which
# session a row came from.
act[COHORTS[0]].head()

## 3. Sanity checks

Four cheap ones. Each compares the data against something stated elsewhere, so
a mismatch names its own cause.

In [ ]:
# TR: the spacing of `time_s` must be the `tr:` in the config. A mismatch here
# means every time column in that cohort is scaled wrong.
cfg = nb.cohort_configs().set_index("cohort")
rows = []
for cohort, df in act.items():
    one = df[(df["sub"] == df["sub"].iloc[0]) & (df["task"] == df["task"].iloc[0])]
    observed = float(np.median(np.diff(one["time_s"].to_numpy())))
    expected = float(cfg.loc[cohort, "tr"])
    rows.append({"cohort": cohort, "tr_config": expected, "tr_observed": round(observed, 4),
                 "match": bool(np.isclose(observed, expected, atol=1e-3))})
pd.DataFrame(rows)

In [ ]:
# Censoring: `good_frame` is false where a frame was scrubbed. Identically 1.0
# for a cohort with censoring off (ds002837 regresses upstream instead), and
# lowest where motion is highest.
pd.DataFrame([{
    "cohort": c,
    "frac_good_frames": round(float(df["good_frame"].mean()), 4),
    "worst_subject": round(float(df.groupby("sub")["good_frame"].mean().min()), 4),
    "subs_below_0.5": int((df.groupby("sub")["good_frame"].mean() < 0.5).sum()),
} for c, df in act.items()])

In [ ]:
# Empty parcels: an all-NaN column is a parcel with no voxels under this
# subject's brain mask -- registration or coverage, not noise. On a coarse
# atlas any of these is worth a look; on harvardoxford a few is normal.
def empty_parcels(df):
    parcels = [c for c in df.columns
               if c not in nb.ACTIVATION_META_COLUMNS
               and c not in ("atlas", "cohort", "task", "sub")]
    per_sub = df.groupby("sub")[parcels].apply(lambda g: g.isna().all().sum())
    return per_sub

rows = []
for cohort, df in act.items():
    e = empty_parcels(df)
    rows.append({"cohort": cohort, "subs_with_empty_parcels": int((e > 0).sum()),
                 "max_empty_parcels": int(e.max()), "n_parcels": len(
                     [c for c in df.columns if c not in nb.ACTIVATION_META_COLUMNS
                      and c not in ("atlas", "cohort", "task", "sub")])})
pd.DataFrame(rows)

In [ ]:
# Run length per subject, which is `frac_stimulus_covered`'s raw material: a
# short run is a scan that stopped early, not a subject who watched faster.
lengths = pd.concat([
    df.groupby(["cohort", "task", "sub"]).size().rename("n_tr").reset_index()
    for df in act.values()
], ignore_index=True)
lengths.groupby("cohort")["n_tr"].describe()[["count", "min", "50%", "max"]]

In [ ]:
if HAVE_MPL:
    fig, ax = plt.subplots(1, len(act), figsize=(4 * len(act), 3), sharey=True)
    for a, (cohort, df) in zip(np.atleast_1d(ax), act.items()):
        a.hist(df.groupby("sub")["good_frame"].mean(), bins=20, range=(0, 1))
        a.set_title(cohort, fontsize=9)
        a.set_xlabel("frac good frames")
    np.atleast_1d(ax)[0].set_ylabel("subjects")
    fig.tight_layout()

## 4. The fine atlas, one subject at a time

`harvardoxford` is 111 parcels. A whole cohort with parcels is a real read, so
price it from the parquet footers before deciding, and take metadata-only or
one subject when you do not need it.

In [ ]:
fine_ds, _ = nb.dataset("activation", ATLAS_FINE)
target = COHORTS[0]
filt = pads.field("cohort") == target

gb_all, n_frag = nb.estimate_gb(fine_ds, filter=filt)
gb_meta, _ = nb.estimate_gb(fine_ds, columns=nb.ACTIVATION_META_COLUMNS, filter=filt)
print(f"{target} @ {ATLAS_FINE}: {n_frag} shard(s)")
print(f"  all columns      {gb_all * 1000:9.2f} MB uncompressed")
print(f"  metadata only    {gb_meta * 1000:9.2f} MB   <- the first-look read")
print(f"  parcel columns   {len(nb.parcel_columns(fine_ds))}")

In [ ]:
# Metadata only: no parcel data is read off disk at all.
meta_fine = nb.load_activation(ATLAS_FINE, cohort=target)
print(len(meta_fine), "rows", round(nb.mem_mb(meta_fine), 1), "MB")

# One subject, with parcels.
sub0 = sorted(meta_fine["sub"].unique())[0]
one = nb.load_activation(ATLAS_FINE, cohort=target, sub=sub0, with_parcels=True)
print(f"sub-{sub0}: {one.shape}  {round(nb.mem_mb(one), 1)} MB")
one.iloc[:3, :8]

## 5. Who these subjects are

Three files, three owners, joined on `sub` (and `task`, where the row is per
run):

| file | owner | carries |
|---|---|---|
| `config/*_participants.csv` | human | curation — `excluded`, `exclusion_reason` |
| `outputs/meta/cohorts/cohort=*/participants_qc.csv` | pipeline | measurement — motion, coverage, scrubbing |
| `config/phenotype/*_phenotype.csv` | human | **age, sex, clinical scores** |
| `outputs/meta/.../participants_scores.csv` | script | Cam-CAN's behavioural battery (`camcan` only) |

The fourth is Cam-CAN's own `cc700-scored/` battery — ten tests, one column
block each — consolidated by
`preprocessing/camcan/03_build_participants_scores.py`. It exists for `camcan`
and not for `camcan_ccfrail`: none of ccfrail's 55 subjects took it.

The third does not exist yet. Age and sex are in none of the files this
pipeline reads or writes — they come from each dataset's own source table
(OpenNeuro's `participants.tsv`, the Cam-CAN archive's scored files) and are
imported once by `tools/make_phenotype.py`. `config/phenotype/README.md` says
where each cohort's lives. Until then `subject_table` says so and carries on,
with `has_pheno=False` marking every row.

**No threshold is applied anywhere below.** `mean_fd` is a measurement;
`mean_fd > 0.5 → exclude` is a claim, and it belongs with the model that rests
on it, so a sensitivity analysis can move it without re-running any of this.

In [ ]:
subjects = {c: nb.subject_table(c) for c in COHORTS}

pd.DataFrame([{
    "cohort": c,
    "rows": len(t),
    "subs": t["sub"].nunique(),
    "excluded": int(t["excluded"].sum()),
    "has_qc": int(t["mean_fd"].notna().sum()) if "mean_fd" in t else 0,
    "has_phenotype": int(t["has_pheno"].sum()),
    "has_scores": int(t["has_scores"].sum()),
    "age": "yes" if "age" in t.columns else "-",
    "sex": "yes" if "sex" in t.columns else "-",
} for c, t in subjects.items()])

In [ ]:
# Motion, per cohort, as measured. ccfrail is the frailty cohort and its
# censoring is heavy -- which is itself a confound, since it removes the most
# data from exactly the participants the study is about.
qc_cols = ["mean_fd", "median_fd", "max_fd", "frac_fd_gt_0p2", "frac_fd_gt_0p5"]
motion = pd.concat([t.assign(cohort=c) for c, t in subjects.items()], ignore_index=True)
present = [c for c in qc_cols if c in motion.columns]
motion.groupby("cohort")[present].describe().T if present else "no QC joined"

In [ ]:
# The analysis-ready subject frame: one row per (cohort, task, sub), curation +
# QC + phenotype + what stage 2 actually produced for them.
per_sub = pd.concat([
    df.groupby(["cohort", "task", "sub"])
      .agg(n_tr=("t", "size"), frac_good=("good_frame", "mean"))
      .reset_index()
    for df in act.values()
], ignore_index=True)

analysis = pd.concat(subjects.values(), ignore_index=True).merge(
    per_sub, on=["cohort", "task", "sub"], how="outer", indicator=True)
print(analysis["_merge"].value_counts().rename("join outcome"))
analysis.head()

## Where this stops

Loaded, checked, joined. What is deliberately not here: no exclusion applied,
no group contrast, no model. Those need the phenotype, and the healthy-vs-frail
contrast in particular needs the acquisition confound handled — `camcan` and
`camcan_ccfrail` differ in TR (2.47 vs 1.12 s) and echoes (5 vs 1), which moves
edge noise in the same direction as the group effect. Stage 3 is where that
becomes measurable; see `02_dfc.ipynb`.

Memory, if you widen any of this:

* keep one dataset object per atlas — schemas differ across them (111 / 14 / 7);
* filter on partition keys (`cohort`, `task`, `sub`) so whole directories are
  pruned before any file opens;
* project columns — `with_parcels=False` reads no parcel data at all;
* price a read with `nb.estimate_gb(...)` before running it;
* anything needing a full-cohort pass over `harvardoxford` belongs in a batch
  job that writes a summary table, not in this notebook.